In [1]:
import numpy as np
from pynq import allocate 
from pynq import Overlay
import time


In [5]:
ol = Overlay("./design_cpy.bit")

In [6]:
# 1. Setup
data_size = 8
input_buffer = allocate(shape=(data_size,), dtype=np.uint32)
output_buffer = allocate(shape=(data_size,), dtype=np.uint32)

dma = ol.axi_dma # Double-check your overlay name
dma_send = dma.sendchannel
dma_recv = dma.recvchannel

# 2. Prepare Data
for i in range(data_size):
    input_buffer[i] = i + 0xcafe0000

# PRE-TRANSFER HYGIENE: 
# Ensure the CPU pushes data from its cache out to the actual DRAM
input_buffer.flush()

print("Starting Transfer...")

# 3. THE CRITICAL ORDER: Start Receive BEFORE Send
# This tells the DMA to "stand by" for data
print("Ouput buffer before run")
for i in range(data_size):
    print(f'Index {i}: {hex(output_buffer[i])}')
    
dma_recv.start() # Attempt to break the hang
# dma_send.start()

dma_recv.transfer(output_buffer)

# Now trigger the source to send the data
# dma_send.transfer(input_buffer)

# 4. SAFE WAIT: Polling instead of .wait()
# This prevents Jupyter from hanging if TLAST never arrives
timeout = 3.0 # seconds
start_time = time.time()

while True:
    # Read the S2MM_DMASR (Status Register)
    # Offset 0x34 is usually the Status Register for Receive
    status = dma.register_map.S2MM_DMASR
    
    # Check if Idle (Bit 1) or if we hit a Timeout
    if status.Idle == 1:
        print("Success: DMA is Idle.")
        break
        
    if (time.time() - start_time) > timeout:
        # If we have the data but it's not 'Idle', the TLAST was missed
        if output_buffer[7] == 0xdeadbeef:
            print("Data received, but TLAST was missed. Forcing Stop...")
            # This is the 'AMD' way to clear the hang:
            dma.register_map.S2MM_DMACR.RS = 0 # Stop the DMA
        else:
            print(f"Total Failure. Status: {hex(status.register_value)}")
        break
    time.sleep(0.1)

# 5. POST-TRANSFER HYGIENE:
# Tell the CPU its cache is old and it must look at the DRAM again
output_buffer.invalidate()

# 6. Results
print("Results:")
for i in range(data_size):
    print(f'Index {i}: {hex(output_buffer[i])}')

Starting Transfer...
Ouput buffer before run
Index 0: 0x0
Index 1: 0x0
Index 2: 0x0
Index 3: 0x0
Index 4: 0x0
Index 5: 0x0
Index 6: 0x0
Index 7: 0x0
Success: DMA is Idle.
Results:
Index 0: 0x48656c6c
Index 1: 0x6f205059
Index 2: 0x4e512120
Index 3: 0xcafebabe
Index 4: 0x1
Index 5: 0x2
Index 6: 0x3
Index 7: 0xdeadbeef
